In [1]:
import pandas as pd
from darts.timeseries import TimeSeries
from darts.utils.timeseries_generation import datetime_attribute_timeseries
from darts.dataprocessing.transformers import Scaler
from darts.models import TFTModel
from darts.dataprocessing.transformers import StaticCovariatesTransformer
import numpy as np
import torch
import matplotlib.pyplot as plt
import joblib
import os

c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md
The `XGBoost` module could not be imported. To enable XGBoost support in Darts, follow the detailed instructions in the installation guide: https://github.com/unit8co/darts/blob/master/INSTALL.md


### Loss Logger 

In [18]:
from pytorch_lightning.callbacks import Callback

class LossLogger(Callback):
    """
    A PyTorch Lightning callback to record training and validation losses 
    at the end of every epoch for custom plotting or analysis.
    """
    def __init__(self):
        super().__init__()
        self.train_losses = []
        self.val_losses = []

    def on_train_epoch_end(self, trainer, pl_module):
        # Retrieve train_loss from callback_metrics
        train_loss = trainer.callback_metrics.get("train_loss")
        
        if train_loss is not None:
            # detach() ensures we don't keep the computation graph in memory
            # cpu() ensures it works regardless of whether you're on GPU or CPU
            self.train_losses.append(float(train_loss.detach().cpu()))

    def on_validation_epoch_end(self, trainer, pl_module):
        # Retrieve val_loss from callback_metrics
        val_loss = trainer.callback_metrics.get("val_loss")
        
        if val_loss is not None:
            self.val_losses.append(float(val_loss.detach().cpu()))

### Add encoders

In [3]:
# Function to encode the year as a normalized value
def encode_year(idx):
  return (idx.year - 2000) / 50

def encode_days_in_month(index):
  return index.days_in_month.to_numpy().reshape(-1,1)

# Set up the add_encoders dictionary to specify how different time-related encoders and transformers should be applied
add_encoders = {
    'cyclic': {'past': ['month'], 'future': ['month']},
    'position': {'past': ['relative'], 'future': ['relative']},
    'custom': {
        'past': [encode_year, encode_days_in_month],
        'future': [encode_year, encode_days_in_month]
    },
    'transformer': Scaler()
}

### Read the data

In [4]:
pandas_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Input_data_preparation\Preparing the input data\Validated_data_for_TFT_training.csv",index_col = ['MONTH_OF_SALE'],parse_dates=True)

In [5]:
pandas_df.groupby("PARENT_DEALER_CODE_MODEL_FAMILY").agg(COUNT_OF_RECORDS=('NET_SALES','count')).sort_values(by='COUNT_OF_RECORDS',ascending=True)

,COUNT_OF_RECORDS
PARENT_DEALER_CODE_MODEL_FAMILY,
11420_GLAMOUR_DISC<>SELF<>CAST<>GREY BLUE,24
11444_XOOM_DRUM<>SELF<>SHEET METAL<>BLUE,24
11445_PLEASURE+_DRUM<>SELF<>CAST<>BLACK,24
11457_GLAMOUR_DISC<>SELF<>CAST<>BLACK,24
11468_DESTINI_DRUM<>SELF<>SHEET METAL<>WHITE,24
...,...
11153_HF DELUXE_DRUM<>SELF<>CAST<>GREY BLACK,36
11153_HF DELUXE_DRUM<>SELF<>CAST<>RED BLACK,36
11153_HF DELUXE_DRUM<>SELF<>CASTI<>HEAVY GREY,36


### Separating the type of variables

In [5]:
#Extracting type of columns according to the datatypes
# 1. Targets/Metrics (The numbers we want to predict)
target_cols = pandas_df.select_dtypes(include=['number']).columns.tolist()
target_cols.append('PARENT_DEALER_CODE_MODEL_FAMILY')

# 2. Time Dimension
time_cols = pandas_df.select_dtypes(include=['datetime', 'datetime64']).columns.tolist()

# 3. Static/Categorical Covariates (The identifiers)
# We exclude numbers and dates to find the "ID" strings
static_cols = pandas_df.select_dtypes(exclude=['number', 'datetime', 'datetime64']).columns.tolist()

print(f"Targets: {target_cols}")
print(f"Time Column: {time_cols}")
print(f"Static Identifiers: {static_cols}")

Targets: ['PARENT_DEALER_CODE', 'NET_SALES', 'DUSSEHRA_(VIJAYADASHAMI)_DAYS', 'NUM_FESTIVE_DAYS_MONTH', 'AKSHAYA_TRITIYA_DAYS', 'BHAI_DOOJ_DAYS', 'BUDDHA_PURNIMA_DAYS', 'CHHATH_PUJA_DAYS', 'DHANTERAS_DAYS', 'DIWALI_DAYS', 'EID_UL_FITR_DAYS', 'GANESH_CHATURTHI_DAYS', 'GANGA_DUSSEHRA_DAYS', 'GOVARDHAN_POOJA_DAYS', 'GURU_PURNIMA_DAYS', 'HANUMAN_JAYANTI_DAYS', 'HARTALIK_TEEJ_DAYS', 'HOLI_DAYS', 'HOLIKA_DAHAN_DAYS', 'JAGANNATH_RATHYATRA_DAYS', 'JANMASHTAMI_DAYS', 'KARWA_CHAUTH_DAYS', 'LOHRI_DAYS', 'MAHA_SHIVARATRI_DAYS', 'MAKAR_SANKRANTI_PONGAL_DAYS', 'NAG_PANCHAMI_DAYS', 'NAVRATRI_DAYS', 'NEW_YEAR_DAYS', 'ONAM_DAYS', 'PITRAPAKSHA_DAYS', 'RAKSHA_BANDHAN_DAYS', 'REPUBLIC_DAY_DAYS', 'VASANT_PANCHAMI_DAYS', 'VISHWAKARMA_PUJA_DAYS', 'MARRIAGE_DAYS', 'FESTIVE_PHASE_I', 'FESTIVE_PHASE_II', 'FESTIVE_PHASE_III', 'PITRU_PAKSH', 'TOTAL_DAYS_FESTIVE_PHASE_I', 'TOTAL_DAYS_FESTIVE_PHASE_II', 'TOTAL_DAYS_FESTIVE_PHASE_III', 'TOTAL_DAYS_PITRU_PAKSH', 'PROP_FESTIVE_PHASE_I', 'PROP_EVENT_FESTIVE_PHASE_I', '

In [6]:
#Separating the covariates
target_col = ["NET_SALES"]

#future covariates
future_covariates = [i for i in target_cols if i!='NET_SALES']

#actual_static_cols
actual_static_cols = [i for i in static_cols if i!='PARENT_DEALER_CODE_MODEL_FAMILY']

In [7]:
actual_static_cols

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME']

In [8]:
# #since variables like MODEL_FAMILY,BRAKE_VARIANT,IGNITION_TYPE,WHEEL_TYPE,BIKE_COLOUR are mostly same for all the top 10 series, will be removing them from the static covariates'
# static_covariates = [i for i in actual_static_cols if i not in ['MODEL_FAMILY','BRAKE_VARIANT','IGNITION_TYPE','WHEEL_TYPE','BIKE_COLOUR','DEALER_CODE']]
# static_covariates

static_covariates = actual_static_cols.copy()

static_covariates

['MODEL_FAMILY',
 'BRAKE_TYPE',
 'IGNITION_TYPE',
 'WHEEL_TYPE',
 'COLOUR',
 'DEALER_CITY',
 'X_CITY_CATEGORY',
 'ZONAL_OFFICE_NAME']

### Preparing data for Darts

In [9]:
#Step 1 - Sorting the dataframe by date
pandas_df.reset_index().sort_values(by=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"]).set_index("MONTH_OF_SALE",inplace=True)

In [10]:
#Step 2 - Separating the static covariates and NET_SALES column
pandas_df_with_target_and_static_covariates = pandas_df.loc[:,['PARENT_DEALER_CODE_MODEL_FAMILY','NET_SALES']+static_covariates]
pandas_df_with_target_and_static_covariates.head()

,PARENT_DEALER_CODE_MODEL_FAMILY,NET_SALES,MODEL_FAMILY,BRAKE_TYPE,IGNITION_TYPE,WHEEL_TYPE,COLOUR,DEALER_CITY,X_CITY_CATEGORY,ZONAL_OFFICE_NAME
MONTH_OF_SALE,,,,,,,,,,
2025-03-01,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY,0.0,XTREME 160,DRUM,SELF,CAST,GREY,PATNA,URBAN,Zonal Office - East
2025-04-01,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY,0.0,XTREME 160,DRUM,SELF,CAST,GREY,PATNA,URBAN,Zonal Office - East
2026-08-01,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY,0.0,XTREME 160,DRUM,SELF,CAST,GREY,PATNA,URBAN,Zonal Office - East
2026-09-01,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY,0.0,XTREME 160,DRUM,SELF,CAST,GREY,PATNA,URBAN,Zonal Office - East
2026-10-01,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY,0.0,XTREME 160,DRUM,SELF,CAST,GREY,PATNA,URBAN,Zonal Office - East


In [11]:
#Step 3 - Separating the future covariates
pandas_df_with_future_covariates = pandas_df.loc[:,future_covariates]
pandas_df_with_future_covariates.head()

,PARENT_DEALER_CODE,DUSSEHRA_(VIJAYADASHAMI)_DAYS,NUM_FESTIVE_DAYS_MONTH,AKSHAYA_TRITIYA_DAYS,BHAI_DOOJ_DAYS,BUDDHA_PURNIMA_DAYS,CHHATH_PUJA_DAYS,DHANTERAS_DAYS,DIWALI_DAYS,EID_UL_FITR_DAYS,...,TOTAL_DAYS_PITRU_PAKSH,PROP_FESTIVE_PHASE_I,PROP_EVENT_FESTIVE_PHASE_I,PROP_FESTIVE_PHASE_II,PROP_EVENT_FESTIVE_PHASE_II,PROP_FESTIVE_PHASE_III,PROP_EVENT_FESTIVE_PHASE_III,PROP_PITRU_PAKSH,PROP_EVENT_PITRU_PAKSH,PARENT_DEALER_CODE_MODEL_FAMILY
MONTH_OF_SALE,,,,,,,,,,,,,,,,,,,,,
2025-03-01,17070,0.0,0.0,0,0,0,0,0,0,1,...,0.0,0.00000,0.0,0.00000,0.00000,0.0,0.0,0.00000,0.00000,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY
2025-04-01,17070,0.0,0.0,1,0,0,0,0,0,0,...,0.0,0.00000,0.0,0.00000,0.00000,0.0,0.0,0.00000,0.00000,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY
2026-08-01,17070,0.0,0.0,0,0,0,0,0,0,0,...,0.0,0.00000,0.0,0.00000,0.00000,0.0,0.0,0.00000,0.00000,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY
2026-09-01,17070,0.0,30.0,0,0,0,0,0,0,0,...,14.0,0.00000,0.0,0.00000,0.00000,0.0,0.0,0.13333,0.28571,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY
2026-10-01,17070,1.0,31.0,0,0,0,0,0,0,0,...,14.0,0.32258,1.0,0.35484,0.73333,0.0,0.0,0.32258,0.71429,17070<>XTREME 160<>DRUM<>SELF<>CAST<>GREY


In [12]:
#Step 4 - Creating the darts timeseries object for target and static covariates
darts_df_with_static_covariates = TimeSeries.from_group_dataframe(df=pandas_df_with_target_and_static_covariates,
                                                                  group_cols=["PARENT_DEALER_CODE_MODEL_FAMILY"],
                                                                  static_cols=static_covariates,value_cols=["NET_SALES"],freq='MS')


In [13]:
#Step 5 - Creating the darts timeseries object with future covariates

#Removing PARENT_DEALER_CODE_MODEL_FAMILY from future_covariates
try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass

darts_df_with_future_covariates = TimeSeries.from_group_dataframe(df = pandas_df_with_future_covariates,
                                    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
                                    freq = 'MS',
                                    value_cols = future_covariates
                                    )

### Train/Test split

In [14]:
#Step 6 - Creating train, test, and validation split
#Train set - Apr'23 to Dec'25 
#Val set - Jan'26 to Mar'26 


train_list = []
val_list = []

for ts in darts_df_with_static_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2026-01-01'), pd.Timestamp('2026-03-01'))
    
    train_list.append(train)
    val_list.append(val)

train_future_covariates_list = []
validation_future_covariates_list = []

for ts in darts_df_with_future_covariates:
    train = ts.slice(pd.Timestamp('2023-04-01'), pd.Timestamp('2025-12-01'))
    val = ts.slice(pd.Timestamp('2026-01-01'), pd.Timestamp('2026-03-01'))
    train_future_covariates_list.append(train)
    validation_future_covariates_list.append(val)

### Scaling

In [15]:

target_scaler = Scaler(n_jobs=-1)
future_covariates_scaler = Scaler(n_jobs=-1)

transformer = StaticCovariatesTransformer(n_jobs=-1)

#Scale the target training data
scaled_target_series = target_scaler.fit_transform(train_list)

scaled_target_series_with_static_covariates_training = transformer.fit_transform(scaled_target_series)



# #Scale the static covariates in training data
# scaled_static_covariates_training = transformer.fit_transform(train_list)

# #Scale the future covariates in training data
# # scaled_future_covariates = future_covariates_scaler.fit_transform(darts_df_with_future_covariates)

scaled_future_covariates_training = future_covariates_scaler.fit_transform(train_future_covariates_list)
scaled_future_covariates_validation = future_covariates_scaler.transform(validation_future_covariates_list)


# #Scale the target validation data
scaled_target_series_validation = target_scaler.transform(val_list)
scaled_target_series_with_static_covariates_validation = transformer.transform(scaled_target_series_validation)

# #Scale the static covariates in validation data
# scaled_static_covariates_validation = transformer.transform(val_list)



In [16]:
from datetime import datetime

from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping
from darts.models import TFTModel

In [22]:
loss_logger=LossLogger()


# =========================
# 2. DEFINE PATHS & MODEL NAME
# =========================
now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")

WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model"
MODEL_NAME = f"tft_net_sales_{now}"

MODEL_DIR = os.path.join(WORK_DIR, MODEL_NAME)
CHECKPOINT_DIR = os.path.join(MODEL_DIR, "checkpoints")

# Create directories if not present
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

print("MODEL_DIR:", MODEL_DIR)
print("CHECKPOINT_DIR:", CHECKPOINT_DIR)


# =========================
# 3. CUSTOM CHECKPOINT CLASS
# =========================
class DateStampedCheckpoint(ModelCheckpoint):

    @property
    def state_key(self) -> str:
        return f"DateStampedCheckpoint_{self.monitor}_{self.dirpath}"


checkpoint_callback = DateStampedCheckpoint(
    dirpath=CHECKPOINT_DIR,
    filename=f"tft-best-{now}-{{epoch:02d}}-{{val_loss:.4f}}",
    monitor="val_loss",
    mode="min",
    save_top_k=1,
    save_last=True,
    verbose=True
)


# =========================
# 4. EARLY STOPPING
# =========================
early_stop_callback = EarlyStopping(
    monitor="val_loss",
    patience=10,
    mode="min",
    verbose=True
)


# =========================
# 5. MODEL DEFINITION
# =========================
model = TFTModel(
    input_chunk_length=12,
    output_chunk_length=3,
    batch_size=512,
    dropout=0.1,
    likelihood=None,
    loss_fn=torch.nn.MSELoss(),
    n_epochs=100,
    random_state=42,
    add_encoders=add_encoders,

    model_name=MODEL_NAME,
    work_dir=WORK_DIR,

    # IMPORTANT: avoid conflict with Darts internal checkpointing
    save_checkpoints=False,
    force_reset=True,

    pl_trainer_kwargs={
        "callbacks": [
            loss_logger,
            checkpoint_callback,
            early_stop_callback
        ],
        "enable_checkpointing": True,
        "gradient_clip_val": 0.1
    }
)


# =========================
# 6. LR FINDER
# =========================
print("\nRunning LR Finder...")

lr_finder = model.lr_find(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training,
)

suggested_lr = lr_finder.suggestion()
print("Suggested Learning Rate:", suggested_lr)

# Apply LR
model.lr = suggested_lr

MODEL_DIR: C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\tft_net_sales_2026-05-15_16_58_29
CHECKPOINT_DIR: C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\tft_net_sales_2026-05-15_16_58_29\checkpoints

Running LR Finder...


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Finding best initial lr: 100%|██████████| 100/100 [00:58<00:00,  1.70it/s]
Restoring states from the checkpoint path at c:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\.lr_find_afdfb601-e840-433a-a66c-6faa4266bdd3.ckpt
Restored all states from the checkpoint at c:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Pipeline\Model\.lr_find_afdfb601-e840-433a-a66c-6faa4266bdd3.ckpt


Suggested Learning Rate: 0.0013182567385564075


In [ ]:
# =========================
# 7. TRAINING WITH VALIDATION
# =========================
print("\nStarting Training...")

model.fit(
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=scaled_future_covariates_training,
    val_series=scaled_target_series_validation,
    val_future_covariates=scaled_future_covariates_validation,
    verbose=True
)



Starting Training...


IndexError: index 0 is out of bounds for axis 0 with size 0

In [ ]:
# =========================
# 8. LOAD BEST MODEL
# =========================
best_model_path = checkpoint_callback.best_model_path

print("\nBest Model Path:", best_model_path)

best_model = TFTModel.load_from_checkpoint(best_model_path)


In [ ]:
# =========================
# 9. USE FOR PREDICTION
# =========================
forecast = best_model.predict(n=12)

### Preparing the prediction data

In [ ]:
# prediction_df = pd.read_csv(r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Iteration 1\Historical_Data_with_Prediction_Data.csv")
prediction_df = pandas_df.loc['2026-04-01':'2026-06-01',:]
# prediction_df.drop_duplicates(subset=["PARENT_DEALER_CODE_MODEL_FAMILY","MONTH_OF_SALE"],inplace=True)

# prediction_df["DEALER_CODE"] = prediction_df["DEALER_CODE"].astype('str')
# prediction_df['MONTH_OF_SALE'] = pd.to_datetime(prediction_df['MONTH_OF_SALE'])

# prediction_df.set_index('MONTH_OF_SALE',inplace=True)

prediction_df.head()


In [17]:
#Making sure prediction_df has the same series as that of training data
unique_series_in_training_data = pandas_df["PARENT_DEALER_CODE_MODEL_FAMILY"].unique().tolist()

unique_series_in_prediction_data = prediction_df["PARENT_DEALER_CODE_MODEL_FAMILY"].unique().tolist()

print(f"{len(set(unique_series_in_prediction_data) - set(unique_series_in_training_data))} series are available in prediction data but not in training data")

print(f"{len(set(unique_series_in_training_data) - set(unique_series_in_prediction_data))} series are available in training data but not in prediction data")

0 series are available in prediction data but not in training data
0 series are available in training data but not in prediction data


### Darts preparation for Prediction Data

In [18]:
prediction_df.index

DatetimeIndex(['2025-04-01', '2025-04-01', '2025-04-01', '2025-04-01',
               '2025-04-01', '2025-04-01', '2025-04-01', '2025-04-01',
               '2025-04-01', '2025-04-01',
               ...
               '2027-03-01', '2027-03-01', '2027-03-01', '2027-03-01',
               '2027-03-01', '2027-03-01', '2027-03-01', '2027-03-01',
               '2027-03-01', '2027-03-01'],
              dtype='datetime64[ns]', name='MONTH_OF_SALE', length=1602600, freq=None)

In [19]:
prediction_pandas_df = prediction_df.copy()

try:
    future_covariates.remove('PARENT_DEALER_CODE_MODEL_FAMILY')
except:
    pass


darts_prediction_series = TimeSeries.from_group_dataframe(
    df=prediction_df,
    group_cols="PARENT_DEALER_CODE_MODEL_FAMILY",
    static_cols=static_covariates, 
    value_cols=future_covariates,
    freq='MS'
)

### Scale the prediction data

In [20]:
scaled_temporal = future_covariates_scaler.transform(darts_prediction_series)

final_prediction_series = transformer.transform(scaled_temporal)

### Train the model

In [27]:
from datetime import datetime
from pytorch_lightning.callbacks import ModelCheckpoint

# now = datetime.now().strftime("%Y-%m-%d_%H_%M_%S")

# MODEL_NAME = f"tft_net_sales_whole_dataset_{now}"

# WORK_DIR = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Iteration 1"

In [28]:
class DateStampedCheckpoint(ModelCheckpoint):
    """
    A ModelCheckpoint subclass with a unique state_key so it can coexist
    with Darts's internal ModelCheckpoint without raising a RuntimeError.
    Saves a best checkpoint with the training date, epoch and val_loss
    baked into the filename.
    """
    @property
    def state_key(self) -> str:
        return f"DateStampedCheckpoint_{self.monitor}_{self.dirpath}"

In [55]:
custom_checkpoint_callback = DateStampedCheckpoint(
    dirpath=rf"{WORK_DIR}\{MODEL_NAME}\checkpoints",
    filename=f"tft-final-model-{now}-{{epoch:02d}}",
    save_top_k=1,
    monitor="epoch", 
    mode="max"       
)

In [56]:
loss_logger = LossLogger()
model = TFTModel(input_chunk_length=12,output_chunk_length=12,
                 batch_size=512,dropout=0.1,likelihood=None,loss_fn=torch.nn.MSELoss(),
                 n_epochs=100,random_state=42,add_encoders=add_encoders,model_name=MODEL_NAME,
                 work_dir = WORK_DIR,save_checkpoints = True,force_reset=True,
                 pl_trainer_kwargs={'callbacks': [loss_logger,custom_checkpoint_callback],
                                    'enable_checkpointing':True})

model.fit(series=scaled_target_series_with_static_covariates_training,
          future_covariates = scaled_future_covariates_training,
          verbose=True)

# trainer = model.trainer
# optimizer = trainer.optimizers[0]
# current_lr = optimizer.param_groups[0]["lr"]
# total_epochs = model.trainer.max_epochs

# def title_of_the_plot():
#     total_epochs = len(loss_logger.losses)
    
#     if total_epochs > 0:
#         last_loss = loss_logger.losses[-1]
#         return f"Overfitting Debug - lr: {current_lr}, epochs: {total_epochs}, last_loss: {last_loss:.2f}"
#     return "Overfitting Debug - No Data"

# plt.figure(figsize=(10, 6))
# plt.plot(loss_logger.losses)
# plt.xlabel('Epoch')
# plt.ylabel('Train Loss')
# plt.title(title_of_the_plot())
# plt.grid(True, alpha=0.3) 
# plt.show()

Epoch 0:   0%|          | 0/1723 [1:34:50<?, ?it/s]


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

   | Name                              | Type                             | Params | Mode 
------------------------------------------------------------------------------------------------
0  | criterion                         | MSELoss                          | 0      | train
1  | train_criterion                   | MSELoss                          | 0      | train
2  | val_criterion                     | MSELoss                          | 0      | train
3  | train_metrics                     | MetricCollection                 | 0      | train
4  | val_metrics                       | MetricCollection                 | 0      | train
5  | input_embeddings                  | _MultiEmbedding                  | 0      | train
6  | static_covariates_vsn             | _VariableSelectionNetwork        | 1.7 K  | train
7  | encoder_vsn

Epoch 0: 100%|██████████| 1696/1696 [21:15<00:00,  1.33it/s, train_loss=0.0356]

c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\callbacks\model_checkpoint.py:388: `ModelCheckpoint(monitor='val_loss')` could not find the monitored key in the returned metrics: ['train_loss', 'epoch', 'step']. HINT: Did you call `log('val_loss', value)` in the `LightningModule`?


Epoch 6:  31%|███       | 520/1696 [06:38<15:01,  1.30it/s, train_loss=0.0389] 


Detected KeyboardInterrupt, attempting graceful shutdown ...


NameError: name 'exit' is not defined

### Prediction using the last checkpoint

In [51]:
BASE_PATH = r"C:\Users\G0004878\Desktop\TFT_Data\Multi_series\12_month_forecast\Iteration 1"
FOLDER_NAME = "tft_net_sales_whole_dataset_2026-05-07_16_48_42"

In [52]:
MODEL_FOLDER = "tft_net_sales_whole_dataset_2026-05-07_16_48_42"

In [53]:
loaded_model = TFTModel.load_from_checkpoint(
    model_name=FOLDER_NAME,  
    work_dir=BASE_PATH,
    best=False               # Important: we want to use the specific checkpoint file
)

print("Model loaded successfully.")

Model loaded successfully.


In [33]:
scaled_target_series_with_static_covariates_training[0]

,NET_SALES
MONTH_OF_SALE,
2023-04-01,0.500
2023-05-01,0.750
2023-06-01,0.500
2023-07-01,0.625
2023-08-01,0.375
...,...
2025-11-01,0.250
2025-12-01,0.125
2026-01-01,0.625


In [36]:
scaled_future_covariates_training[0]

,DEALER_CODE,NUM_FESTIVE_DAYS_MONTH,FESTIVE_PHASE_I,FESTIVE_PHASE_II,FESTIVE_PHASE_III,...,RAKSHA_BANDHAN_DAYS,REPUBLIC_DAY_DAYS,VASANT_PANCHAMI_DAYS,VISHWAKARMA_PUJA_DAYS,MARRIAGE_DAYS
MONTH_OF_SALE,,,,,,,,,,,
2023-04-01,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000
2023-05-01,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.857143
2023-06-01,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.714286
2023-07-01,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.000000
2023-08-01,0.0,0.000000,0.0,0.0,0.0,...,1.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
2025-11-01,0.0,0.967742,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.714286
2025-12-01,0.0,0.000000,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.142857
2026-01-01,0.0,0.000000,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.714286


In [34]:
final_prediction_series[0]

,DEALER_CODE,NUM_FESTIVE_DAYS_MONTH,FESTIVE_PHASE_I,FESTIVE_PHASE_II,FESTIVE_PHASE_III,...,RAKSHA_BANDHAN_DAYS,REPUBLIC_DAY_DAYS,VASANT_PANCHAMI_DAYS,VISHWAKARMA_PUJA_DAYS,MARRIAGE_DAYS
MONTH_OF_SALE,,,,,,,,,,,
2026-04-01,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.928571
2026-05-01,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.357143
2026-06-01,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.428571
2026-07-01,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.142857
2026-08-01,0.0,0.000000,0.0,0.000000,0.0,...,1.0,0.0,0.0,0.0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...
2026-11-01,0.0,0.967742,0.0,0.266667,1.0,...,0.0,0.0,0.0,0.0,0.285714
2026-12-01,0.0,0.000000,0.0,0.000000,0.0,...,0.0,0.0,0.0,0.0,0.214286
2027-01-01,0.0,0.000000,0.0,0.000000,0.0,...,0.0,1.0,0.0,0.0,0.000000


In [54]:
pred_series = loaded_model.predict(
    n=12,
    series=scaled_target_series_with_static_covariates_training,
    future_covariates=final_prediction_series
)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
You are using a CUDA device ('NVIDIA RTX 2000 Ada Generation') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
c:\Users\G0004878\Desktop\Virtual_environments\darts_gpu\lib\site-packages\pytorch_lightning\trainer\connectors\data_connector.py:425: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=19` in the `DataLoader` to improve performance.


Predicting DataLoader 0: 100%|██████████| 131/131 [01:00<00:00,  2.18it/s]


In [55]:
pred_series_inv = target_scaler.inverse_transform(pred_series)

In [56]:
# Build output DataFrame — one row per series per month
records = []

# 1. Use your final_forecast (inverse transformed) 
# 2. Use your training_series (to extract IDs)
for forecast, original_series in zip(pred_series_inv, scaled_target_series_with_static_covariates_training):

    # Get the series label (Dealer/Model Family)
    series_name = original_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    forecast_values = forecast.values().flatten()

    for month, pred in zip(months, forecast_values):
        records.append({
            'MONTH_OF_SALE'                   : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY' : series_name,
            'PREDICTED_SALES'                 : round(float(pred), 2)
        })

df_output = pd.DataFrame(records)

# Formatting
df_output['MONTH_OF_SALE'] = pd.to_datetime(df_output['MONTH_OF_SALE']).dt.strftime('%Y-%m-%d')
df_output = df_output.sort_values(['PARENT_DEALER_CODE_MODEL_FAMILY', 'MONTH_OF_SALE']).reset_index(drop=True)

# Diagnostics
print(f'Output shape : {df_output.shape}')
print(f'Forecast Months : {df_output["MONTH_OF_SALE"].unique()}')
print(f'Series count : {df_output["PARENT_DEALER_CODE_MODEL_FAMILY"].nunique()}')

Output shape : (801300, 3)
Forecast Months : ['2026-04-01' '2026-05-01' '2026-06-01' '2026-07-01' '2026-08-01'
 '2026-09-01' '2026-10-01' '2026-11-01' '2026-12-01' '2027-01-01'
 '2027-02-01' '2027-03-01']
Series count : 66775


In [ ]:
records = []

for forecast, raw_series in zip(pred_series_inv, darts_df_with_static_covariates):

    series_name = raw_series.static_covariates['PARENT_DEALER_CODE_MODEL_FAMILY'].values[0]

    months          = forecast.time_index
    forecast_values = forecast.values().flatten()

    for month, pred in zip(months, forecast_values):
        records.append({
            'MONTH_OF_SALE'                   : month,
            'PARENT_DEALER_CODE_MODEL_FAMILY' : series_name,
            'PREDICTED_SALES'                 : round(float(pred), 2)
        })

final_output = pd.DataFrame(records)

In [63]:
final_output.head()

,MONTH_OF_SALE,PARENT_DEALER_CODE_MODEL_FAMILY,PREDICTED_SALES
0,2026-04-01,10001_DESTINI_DRUM<>SELF<>CAST<>BLACK,2.70
1,2026-05-01,10001_DESTINI_DRUM<>SELF<>CAST<>BLACK,2.12
2,2026-06-01,10001_DESTINI_DRUM<>SELF<>CAST<>BLACK,1.92
3,2026-07-01,10001_DESTINI_DRUM<>SELF<>CAST<>BLACK,1.70
4,2026-08-01,10001_DESTINI_DRUM<>SELF<>CAST<>BLACK,1.51


In [64]:
final_output["PREDICTED_SALES"].sum()

4701988.380000001

In [65]:
final_output.loc[final_output["MONTH_OF_SALE"]=='2026-04-01',:]["PREDICTED_SALES"].sum()

406184.74

In [ ]:
final_output.to_csv(r"Final_prediction_data.csv")